# Timeline plan: era definitions and bucketing strategy

Translates the research findings into a concrete recipe. No data is scraped here — this notebook only fixes the *definitions* used by `03_execute.ipynb`.

## Era buckets

We use **death year** as the primary anchor (the user's call — a character's era is the era they died in). For characters still alive in the books we use **birth year + 30** as a proxy for their active era. Years are AC-based integers (BC years are negative).

| Era | Year window | Anchor events | Examples |
|---|---|---|---|
| **Pre-Conquest** | `< 1 AC` | Andal invasion, Long Night, Age of Heroes | Bran the Builder, Lann the Clever |
| **Targaryen Conquest** | `1 AC – 50 AC` | Aegon's Conquest, Faith Militant | Aegon I, Visenya, Maegor |
| **Fire & Blood** | `50 AC – 200 AC` | Dance of the Dragons, Regency | Jaehaerys I, Viserys I, Rhaenyra, Aegon III |
| **Mid-Targaryen** | `200 AC – 250 AC` | Blackfyre Rebellions, Dunk & Egg | Daeron II, Aegon V, Duncan the Tall |
| **Late Targaryen** | `250 AC – 282 AC` | War of the Ninepenny Kings | Aerys II, Tywin, Rhaegar |
| **Robert's Reign** | `283 AC – 297 AC` | Robert's Rebellion, Greyjoy Rebellion | Robert, Ned, Cersei (active) |
| **ASOIAF main** | `298 AC – 305 AC` | War of the Five Kings | Tyrion, Daenerys, Jon, Arya |
| **Unknown** | — | — | residual |

Boundaries are deliberately coarse — they need to survive the natural noise in scraped dates (estate biographers often disagree by ± 2 years).

**Why these specific cutoffs:**
- 50 AC: end of Maegor's reign, beginning of Jaehaerys I's golden age. Fire & Blood ends officially at 136 AC with Aegon III's regency, but 50 AC gives a cleaner Targaryen-conquest period.
- 200 AC: convenient round number bridging F&B's official end and the Dunk & Egg era.
- 282 / 283 AC: split point at Robert's Rebellion — characters who died before lived in the late Targaryen world.
- 298 AC: A Game of Thrones starts at 298 AC. Everyone with ASOIAF book mentions falls in here.

In [1]:
# Canonical era definitions used by 03_execute.ipynb
ERAS = [
    ('Pre-Conquest',        None,  1),
    ('Targaryen Conquest',     1,  50),
    ('Fire & Blood',          50, 200),
    ('Mid-Targaryen',        200, 250),
    ('Late Targaryen',       250, 283),
    ("Robert's Reign",      283, 298),
    ('ASOIAF main',          298, 310),
    ('Unknown',             None, None),
]


def assign_era(year):
    """Map a year (int, AC-based, BC negative) to an era label."""
    if year is None:
        return 'Unknown'
    for name, lo, hi in ERAS:
        if name == 'Pre-Conquest' and year < hi:
            return name
        if lo is not None and hi is not None and lo <= year < hi:
            return name
    return 'Unknown'


# Sanity check
for y in [-100, -2, 1, 37, 130, 209, 283, 298, 305]:
    print(f'  Year {y:>4d} AC  ->  {assign_era(y)}')


  Year -100 AC  ->  Pre-Conquest
  Year   -2 AC  ->  Pre-Conquest
  Year    1 AC  ->  Targaryen Conquest
  Year   37 AC  ->  Targaryen Conquest
  Year  130 AC  ->  Fire & Blood
  Year  209 AC  ->  Mid-Targaryen
  Year  283 AC  ->  Robert's Reign
  Year  298 AC  ->  ASOIAF main
  Year  305 AC  ->  ASOIAF main


## Cascade: how to pick a year for each character

Source priority for the date year, per the research notebook decision:

1. **Infobox `Died`** — use directly
2. **Infobox `Born` + 30** — proxy active year for characters still alive (or those with no Died field)
3. **Has any ASOIAF book h3 subsection** → year = 300 AC (middle of ASOIAF main era)
4. **Bio mentions specific event** — use the year of that event (table in `01_research.ipynb`)
5. **Otherwise** → `None` → `Unknown` era

We track which source produced each year so we can audit downstream: a character labelled ASOIAF main *because* of a book mention is more reliable than one labelled ASOIAF main because of a bio prose match.

## Validation plan

Two checks once eras are assigned:

**1. Coverage.** Bar chart: number of characters per era. Expect ASOIAF main to dominate (it's the focus of the books), Pre-Conquest to be a thin tail, Unknown to be 15–25% of the dataset.

**2. Era × Louvain community heatmap.** This is the payoff: if temporal information explains the smear caveat, the Crownlands Louvain community should split clearly into a Fire & Blood band + an ASOIAF main band when stratified by era. If it doesn't, our era assignment is too noisy to be useful.

**Stretch goal:** drop edges where the source's era and the target's era are different by 2+ buckets, re-run Louvain, compare modularity. If it goes up, the temporal-smear fix is real.